<a href="https://colab.research.google.com/github/AnnaZapototska/goit-colab-homeworks/blob/development/hw_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 1

In [10]:
#import os
#os.kill(os.getpid(), 9)

!pip uninstall -y gensim
!pip install gensim==4.3.3

Found existing installation: gensim 4.3.2
Uninstalling gensim-4.3.2:
  Successfully uninstalled gensim-4.3.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.6/26.6 MB 30.6 MB/s eta 0:00:00


In [4]:
!pip install scipy==1.13.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 949.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 40.7 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.11.4
    Uninstalling scipy-1.11.4:
      Successfully uninstalled scipy-1.11.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
access 1.1.10.post3 requires scipy>=1.14.1, but you have scipy 1.13.1 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompati

In [37]:
import pickle
import gensim.downloader as api
import pandas as pd
import numpy as np
from numpy import dot
from numpy.linalg import norm
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

with open('/content/model/word_embeddings_subset.p', 'rb') as f:
    model = pickle.load(f)

print("Amount of the words in the model: ", len(model))

words = list(model.keys())
vectors = np.array(list(model.values()))

pca = PCA(n_components=3)
vectors_3d = pca.fit_transform(vectors)
print(vectors_3d.shape)

df = pd.DataFrame({
    'word': words,
    'x': vectors_3d[:, 0],
    'y': vectors_3d[:, 1],
    'z': vectors_3d[:, 2]
})

df.head()


Amount of the words in the model:  243
(243, 3)


,word,x,y,z
0,country,0.746037,-0.387964,-0.482691
1,city,0.102494,0.140384,-1.189890
2,China,0.831055,-0.129500,-0.312599
3,Iraq,0.656337,-0.177791,-0.338381
4,oil,0.658345,-0.432464,-0.621213


In [31]:
def find_closest_word(vector, df):

    distances = np.sqrt(
        (df['x'] - vector[0])**2 +
        (df['y'] - vector[1])**2 +
        (df['z'] - vector[2])**2
    )

    closest_index = distances.idxmin()

    return (
        df.loc[closest_index, 'word'],
        distances[closest_index]
    )

# test 1
test_vector = [
    df.iloc[100]['x'],
    df.iloc[100]['y'],
    df.iloc[100]['z']
]

print("Original word:", df.iloc[100]['word'])

print(
    "Nearest word:",
    find_closest_word(test_vector, df)
)

# test 2
test_vector_2 = [
    df.iloc[0]['x'],
    df.iloc[0]['y'],
    df.iloc[10]['z']
]

print("Mixed vector:", df.iloc[10]['word'])

print(
    "Nearest word:",
    find_closest_word(test_vector_2, df)
)

Original word: Croatia
Nearest word: ('Croatia', 0.0)
Mixed vector: Japan
Nearest word: ('oil', 0.17047536)


In [40]:
print("\nCross products for 3D vectors:")
word_pairs = [
    ("country", "town"),
    ("Canada", "oil"),
    ("city", "Australia"),
    ("Germany", "Japan")
]

for w1, w2 in word_pairs:

    vec1 = df[df["word"] == w1][["x", "y", "z"]].values[0]
    vec2 = df[df["word"] == w2][["x", "y", "z"]].values[0]

    cross = np.cross(vec1, vec2)

    closest_word = find_closest_word(cross, df)

    print(f"{w1} × {w2}")
    print("Cross product:", cross)
    print("Closest word:", closest_word)
    print("-" * 40)


Cross products for 3D vectors:
country × town
Cross product: [0.3698789  0.6034324  0.08666616]
Closest word: ('Georgia', 0.4093613)
----------------------------------------
Canada × oil
Cross product: [-0.06753445  0.2520901  -0.24706644]
Closest word: ('Nuuk', 0.37897384)
----------------------------------------
city × Australia
Cross product: [-0.6554764  -1.1631671  -0.19369203]
Closest word: ('Monrovia', 0.5024248)
----------------------------------------
Germany × Japan
Cross product: [-0.3018412   0.65169054 -0.49547428]
Closest word: ('Oslo', 0.21249598)
----------------------------------------


# Summary

The nearest words obtained from the cross products were mostly related to geography. This suggests that even after applying PCA and computing orthogonal vectors, the semantic information of the original words was partially preserved. The experiment shows that word embeddings capture meaningful relationships between words and allow mathematical operations to be performed on semantic representations.

In [41]:
def angle_between_words(word1, word2, model):

    vec1 = np.array(model[word1])
    vec2 = np.array(model[word2])

    dot_product = np.dot(vec1, vec2)
    length_v1 = np.linalg.norm(vec1)
    length_v2 = np.linalg.norm(vec2)

    cosine = dot_product / (length_v1 * length_v2)

    angle = np.degrees(np.arccos(cosine))

    return angle

pairs = [
    ("Germany", "Japan"),
    ("country", "town"),
    ("Canada", "oil"),
    ("city", "Australia")
]

for w1, w2 in pairs:
    angle = angle_between_words(w1, w2, model)

    print(f"{w1} - {w2}: {angle:.2f}°")

Germany - Japan: 58.79°
country - town: 70.93°
Canada - oil: 86.37°
city - Australia: 90.41°


# Summary

A function was implemented to calculate the angle between word vectors using the cosine similarity formula. According to the result of the test words:

1.   Germany and Japan have the most similar destinations among these pairs.
2.   country and town - they are related, but not identical.

The pairs "Canada" and "oil" (86.37°) and "city" and "Australia" (90.41°) produced angles close to 90°, suggesting weak semantic relationships.

